# Exercise F — The Measure → Change → Re-measure Loop
The professional RAG workflow: never ship a change you can't show helped. Pick one knob, score a baseline, change it, re-score, compare.

**Offline scaffold:** a from-scratch faithfulness proxy (word-overlap) stands in for RAGAS so it runs now. Swap in real RAGAS for the real exercise.

In [ ]:
# Offline mock so this scaffold runs with NO API key / NO network.
# For real practice, replace `embed()` with your real embedder (InHouseEmbeddings,
# SentenceTransformer, etc.) and `llm()` with a real model call.
import numpy as np, re
_STOP=set("the a an to of and or is are be for in on at by with from as that this it its".split())
def _tok(t): return [w for w in re.findall(r"[a-z0-9]+",t.lower()) if w not in _STOP and len(w)>2]
def embed(texts):
    if isinstance(texts,str): texts=[texts]
    out=[]
    for t in texts:
        v=np.zeros(256)
        for w in _tok(t): v[abs(hash(w))%256]+=1
        n=np.linalg.norm(v); out.append(v/n if n else v)
    return np.array(out)
def cos(a,b): return float(a@b)

# A small corpus standing in for chunks of ERP-2008-chapter4.pdf (health-care economics).
CORPUS = [
 ("Demand for health care is derived from the value of improved health, not the procedures themselves.","demand"),
 ("Health can be defined by longevity (length of life) and quality of life.","demand"),
 ("National health spending reached over 7000 dollars per capita and about 16 percent of GDP.","spending"),
 ("Medical technology accounts for about half of long-term health spending growth.","spending"),
 ("Medicare, enacted in 1965, covers people aged 65 and older; Part D is the drug benefit.","medicare"),
 ("Medicaid, established in 1965, is a program for low-income individuals, administered by states.","medicaid"),
 ("Moral hazard is the tendency to overuse care when insurance covers most of the cost.","moral_hazard"),
 ("Adverse selection is when insurance is most attractive to those most likely to need it.","insurance"),
 ("Health Savings Accounts use pre-tax dollars with high-deductible plans to reduce routine-care reliance.","hsa"),
 ("The proposed standard deduction for health insurance would be a flat 15000 dollars per family.","tax"),
]
texts=[c[0] for c in CORPUS]; sections=[c[1] for c in CORPUS]
print("Mock corpus ready:", len(texts), "chunks.")

In [ ]:
# A tiny faithfulness proxy: fraction of answer claims supported by the retrieved context.
def faithfulness_proxy(answer, context):
    claims = [c.strip() for c in re.split(r"[.;]| and ", answer) if c.strip()]
    if not claims: return 1.0
    ctx = set(_tok(context))
    supported = sum(1 for c in claims if len(set(_tok(c)) & ctx)/max(1,len(set(_tok(c)))) >= 0.5)
    return supported/len(claims)

def rag_answer(query, k):
    qv = embed(query)[0]
    order = sorted(range(len(texts)), key=lambda i:-cos(qv, embed(texts[i])[0]))[:k]
    context = " ".join(texts[i] for i in order)
    # mock answer = echo the top context (a real LLM would synthesize)
    answer = texts[order[0]]
    return answer, context

QUERY = "how is the demand for health defined?"
print("Sweeping top_k and measuring faithfulness proxy:")
for k in [1, 2, 3, 5]:
    ans, ctx = rag_answer(QUERY, k)
    print(f"  k={k}: faithfulness={faithfulness_proxy(ans, ctx):.2f}")

### Observe & decide
- Watch how the score moves as `k` changes. More context isn't always better — it can dilute faithfulness by adding unrelated claims.
**Your turn:** replace the proxy with real RAGAS `Faithfulness()` and `ResponseRelevancy()` (wired to your in-house models per ragas_guide), pick ONE knob (chunk size / k / alpha), and run a real baseline-vs-candidate comparison. This is the meta-skill that ties the whole course together.